
# Figure 2 — methodological effects (v8)

This notebook generates Figure 2 from the analytical outputs produced by Notebook 1.

This version replaces the nested spacer-row approach with four independent Matplotlib `SubFigure` objects.

Consequences:

- A and C have exactly the same outer dimensions.
- B and D have exactly the same outer dimensions.
- Legends and shared x-axis labels are positioned at the `SubFigure` level.
- Panel C uses one shared x-axis title and a separate legend.
- Panel B uses one shared x-axis title and a separate task legend.
- No auxiliary axes reduce the plotting area of B or C.

Outer layout:

```text
A | C
B | D
```


## 1. Imports and paths

In [ ]:

from __future__ import annotations

from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import hashlib
import json
import platform
import subprocess
import sys

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd


In [ ]:

CURRENT_DIRECTORY = Path.cwd().resolve()
REPO_ROOT = (
    CURRENT_DIRECTORY.parent
    if CURRENT_DIRECTORY.name == "notebooks"
    else CURRENT_DIRECTORY
)
if not (REPO_ROOT / "configs").is_dir():
    raise RuntimeError(
        "Run this notebook from the repository root or notebooks directory."
    )

RESULTS_ROOT = REPO_ROOT / "results"
ANALYSIS_ROOT = RESULTS_ROOT / "final_analysis"
FIGURE2_DATA_ROOT = ANALYSIS_ROOT / "figure2_data"
FIGURE_ROOT = RESULTS_ROOT / "figures"
METADATA_ROOT = REPO_ROOT / "metadata"

PANEL_A_SEED_PATH = (
    FIGURE2_DATA_ROOT
    / "panel_a_negative_class_effect_seed_level.csv"
)
PANEL_A_SUMMARY_PATH = (
    FIGURE2_DATA_ROOT
    / "panel_a_negative_class_effect_summary.csv"
)

PANEL_B_SEED_PATH = (
    FIGURE2_DATA_ROOT
    / "panel_b_generalisation_trajectories_seed_level.csv"
)
PANEL_B_SUMMARY_PATH = (
    FIGURE2_DATA_ROOT
    / "panel_b_generalisation_trajectories_summary.csv"
)

PANEL_C_SEED_PATH = (
    FIGURE2_DATA_ROOT
    / "panel_c_methodological_effects_seed_level.csv"
)
PANEL_C_SUMMARY_PATH = (
    FIGURE2_DATA_ROOT
    / "panel_c_methodological_effects_summary.csv"
)

PANEL_D_SEED_PATH = (
    FIGURE2_DATA_ROOT
    / "panel_d_effect_magnitudes_vs_model_family_seed_level.csv"
)
PANEL_D_SUMMARY_PATH = (
    FIGURE2_DATA_ROOT
    / "panel_d_effect_magnitudes_vs_model_family_summary.csv"
)

FIGURE_PNG_PATH = FIGURE_ROOT / "figure_2_v8.png"
FIGURE_PDF_PATH = FIGURE_ROOT / "figure_2_v8.pdf"
FIGURE_SVG_PATH = FIGURE_ROOT / "figure_2_v8.svg"

FIGURE_MANIFEST_PATH = (
    METADATA_ROOT
    / "figure_2_methodological_effects_v8_manifest.json"
)

FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
METADATA_ROOT.mkdir(parents=True, exist_ok=True)

INPUT_PATHS = {
    "panel_a_seed": PANEL_A_SEED_PATH,
    "panel_a_summary": PANEL_A_SUMMARY_PATH,
    "panel_b_seed": PANEL_B_SEED_PATH,
    "panel_b_summary": PANEL_B_SUMMARY_PATH,
    "panel_c_seed": PANEL_C_SEED_PATH,
    "panel_c_summary": PANEL_C_SUMMARY_PATH,
    "panel_d_seed": PANEL_D_SEED_PATH,
    "panel_d_summary": PANEL_D_SUMMARY_PATH,
}

missing_inputs = [
    path
    for path in INPUT_PATHS.values()
    if not path.is_file()
]

if missing_inputs:
    raise FileNotFoundError(
        "Missing Figure 2 data files:\n"
        + "\n".join(str(path) for path in missing_inputs)
    )

print(f"Figure 2 data root: {FIGURE2_DATA_ROOT}")
print(f"Figure output root: {FIGURE_ROOT}")


## 2. Visual configuration

In [ ]:

TASK_ORDER = ["T1", "T2"]

# Task colours are used only for the two benchmark definitions.
TASK_COLORS = {
    "T1": "#2A9D8F",
    "T2": "#D9657B",
}

REGIME_ORDER = [
    "RANDOM",
    "H90",
    "H70",
    "H50",
    "H30",
]

REGIME_LABELS = {
    "RANDOM": "Random",
    "H90": "SIM90",
    "H70": "SIM70",
    "H50": "SIM50",
    "H30": "SIM30",
}

# Regimes use a neutral sequential palette, distinct from tasks and decisions.
REGIME_COLORS = {
    "RANDOM": "#D1D5DB",
    "H90": "#B8C0CC",
    "H70": "#9DA8B6",
    "H50": "#7F8D9D",
    "H30": "#5F6F82",
}

METRIC_ORDER = [
    "Average precision",
    "MCC",
]

EFFECT_ORDER = [
    "negative_class_construction",
    "partition_strategy",
    "redundancy_threshold",
]

MAGNITUDE_ORDER = [
    "negative_class_construction",
    "partition_strategy",
    "redundancy_threshold",
    "model_family",
]

EFFECT_LABELS = {
    "negative_class_construction": "Negative-class\nconstruction",
    "partition_strategy": "Partition\nstrategy",
    "redundancy_threshold": "Similarity\nthreshold",
    "model_family": "Model\nfamily",
}

# Methodological decisions use a categorical palette independent of tasks/regimes.
EFFECT_COLORS = {
    "negative_class_construction": "#4C78A8",
    "partition_strategy": "#D65A8A",
    "redundancy_threshold": "#D8A72E",
    "model_family": "#8F80C9",
}

TEXT_COLOR = "#2E2E2E"
EDGE_COLOR = "#464646"
GRID_COLOR = "#D9D9D9"
ZERO_COLOR = "#777777"
BACKGROUND_COLOR = "white"

FIGURE_WIDTH_INCHES = 14.0
FIGURE_HEIGHT_INCHES = 9.2
RASTER_DPI = 600

PANEL_LETTER_SIZE = 14.0
PANEL_TITLE_SIZE = 11.3
AXIS_LABEL_SIZE = 9.8
TICK_LABEL_SIZE = 8.5
LEGEND_SIZE = 8.9
ANNOTATION_SIZE = 8.2


In [ ]:

mpl.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": [
            "Arial",
            "Helvetica",
            "DejaVu Sans",
        ],
        "font.size": 8.8,
        "axes.labelsize": AXIS_LABEL_SIZE,
        "axes.titlesize": PANEL_TITLE_SIZE,
        "axes.linewidth": 0.75,
        "axes.edgecolor": EDGE_COLOR,
        "axes.labelcolor": TEXT_COLOR,
        "xtick.labelsize": TICK_LABEL_SIZE,
        "ytick.labelsize": TICK_LABEL_SIZE,
        "xtick.color": TEXT_COLOR,
        "ytick.color": TEXT_COLOR,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.major.size": 3.0,
        "ytick.major.size": 3.0,
        "legend.fontsize": LEGEND_SIZE,
        "legend.frameon": False,
        "text.color": TEXT_COLOR,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "savefig.transparent": False,
        "figure.facecolor": BACKGROUND_COLOR,
        "axes.facecolor": BACKGROUND_COLOR,
    }
)


## 3. Load and validate data

In [ ]:

panel_a_seed = pd.read_csv(PANEL_A_SEED_PATH)
panel_a_summary = pd.read_csv(PANEL_A_SUMMARY_PATH)

panel_b_seed = pd.read_csv(PANEL_B_SEED_PATH)
panel_b_summary = pd.read_csv(PANEL_B_SUMMARY_PATH)

panel_c_seed = pd.read_csv(PANEL_C_SEED_PATH)
panel_c_summary = pd.read_csv(PANEL_C_SUMMARY_PATH)

panel_d_seed = pd.read_csv(PANEL_D_SEED_PATH)
panel_d_summary = pd.read_csv(PANEL_D_SUMMARY_PATH)

for dataframe in [
    panel_a_seed,
    panel_b_seed,
    panel_c_seed,
    panel_d_seed,
]:
    dataframe["value"] = pd.to_numeric(
        dataframe["value"],
        errors="raise",
    )

for dataframe in [
    panel_a_summary,
    panel_b_summary,
    panel_c_summary,
    panel_d_summary,
]:
    for column in [
        "median",
        "q1",
        "q3",
        "bootstrap_median_ci_low",
        "bootstrap_median_ci_high",
    ]:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="raise",
        )

print("Loaded and validated Figure 2 datasets.")


## 4. Shared plotting utilities

In [ ]:

def clean_axis(
    axis: plt.Axes,
    *,
    grid_axis: str | None = "y",
) -> None:
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.spines["left"].set_linewidth(0.75)
    axis.spines["bottom"].set_linewidth(0.75)

    if grid_axis is not None:
        axis.set_axisbelow(True)
        axis.grid(
            axis=grid_axis,
            color=GRID_COLOR,
            linewidth=0.55,
            alpha=0.80,
        )


def add_panel_header(
    axis: plt.Axes,
    *,
    letter: str,
    title: str,
) -> None:
    axis.text(
        -0.12,
        1.10,
        letter,
        transform=axis.transAxes,
        fontsize=PANEL_LETTER_SIZE,
        fontweight="bold",
        ha="left",
        va="bottom",
        clip_on=False,
    )

    axis.text(
        0.0,
        1.10,
        title,
        transform=axis.transAxes,
        fontsize=PANEL_TITLE_SIZE,
        fontweight="bold",
        ha="left",
        va="bottom",
        clip_on=False,
    )


def padded_limits(
    values: np.ndarray,
    *,
    include_zero: bool = False,
    lower_bound: float | None = None,
    upper_bound: float | None = None,
    fraction: float = 0.10,
) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if values.size == 0:
        raise ValueError("Cannot determine limits from empty data.")

    minimum = float(np.min(values))
    maximum = float(np.max(values))

    if include_zero:
        minimum = min(minimum, 0.0)
        maximum = max(maximum, 0.0)

    span = maximum - minimum

    if span == 0.0:
        span = max(abs(maximum), 1.0) * 0.15

    lower = minimum - span * fraction
    upper = maximum + span * fraction

    if lower_bound is not None:
        lower = max(lower, lower_bound)

    if upper_bound is not None:
        upper = min(upper, upper_bound)

    return lower, upper


def draw_interval(
    axis: plt.Axes,
    *,
    y: float,
    median: float,
    q1: float,
    q3: float,
    ci_low: float,
    ci_high: float,
    color: str,
) -> None:
    axis.plot(
        [ci_low, ci_high],
        [y, y],
        color=color,
        linewidth=1.2,
        alpha=0.70,
        solid_capstyle="round",
        zorder=2,
    )

    axis.plot(
        [q1, q3],
        [y, y],
        color=color,
        linewidth=4.7,
        alpha=0.95,
        solid_capstyle="round",
        zorder=3,
    )

    axis.scatter(
        [median],
        [y],
        s=50,
        facecolor=color,
        edgecolor=EDGE_COLOR,
        linewidth=0.55,
        zorder=4,
    )


def format_effect_value(
    value: float,
) -> str:
    if abs(value) >= 0.1:
        return f"{value:.2f}"

    return f"{value:.3f}"


def visible_metric_name(
    metric_label: str,
) -> str:
    return (
        "Average precision"
        if metric_label == "AP"
        else "MCC"
    )


def set_metric_title(
    axis: plt.Axes,
    metric_label: str,
) -> None:
    axis.set_title(
        visible_metric_name(metric_label),
        loc="right",
        pad=6,
        fontsize=9.3,
        fontweight="bold",
    )


## 5. Panel-specific plotting functions

In [ ]:

def plot_panel_a(
    axis: plt.Axes,
    *,
    metric_label: str,
) -> None:
    metric_seed = panel_a_seed[
        panel_a_seed["metric_label"].eq(metric_label)
    ].copy()

    x = np.arange(
        len(REGIME_ORDER),
        dtype=float,
    )

    values_by_regime = []

    for regime_id in REGIME_ORDER:
        values = metric_seed.loc[
            metric_seed["regime_id"].eq(regime_id),
            "value",
        ].to_numpy(dtype=float)

        if values.size == 0:
            raise ValueError(
                f"Panel A lacks values for {metric_label}/{regime_id}."
            )

        values_by_regime.append(values)

    violin = axis.violinplot(
        values_by_regime,
        positions=x,
        widths=0.72,
        showmeans=False,
        showextrema=False,
        showmedians=False,
    )

    for body, regime_id in zip(
        violin["bodies"],
        REGIME_ORDER,
    ):
        body.set_facecolor(
            REGIME_COLORS[regime_id]
        )
        body.set_edgecolor(EDGE_COLOR)
        body.set_linewidth(0.60)
        body.set_alpha(0.72)

    # Embedded narrow boxplots provide the distribution summary.
    boxplot = axis.boxplot(
        values_by_regime,
        positions=x,
        widths=0.18,
        patch_artist=True,
        showfliers=False,
        manage_ticks=False,
        whis=(5, 95),
        medianprops={
            "color": EDGE_COLOR,
            "linewidth": 1.25,
        },
        boxprops={
            "facecolor": "white",
            "edgecolor": EDGE_COLOR,
            "linewidth": 0.75,
            "alpha": 0.92,
        },
        whiskerprops={
            "color": EDGE_COLOR,
            "linewidth": 0.70,
        },
        capprops={
            "color": EDGE_COLOR,
            "linewidth": 0.70,
        },
    )

    axis.axhline(
        0.0,
        color=ZERO_COLOR,
        linewidth=0.8,
        linestyle=(0, (4, 3)),
        zorder=0,
    )

    axis.set_xticks(x)
    axis.set_xticklabels(
        [
            REGIME_LABELS[regime_id]
            for regime_id in REGIME_ORDER
        ],
        rotation=24,
        ha="right",
    )

    axis.set_xlabel(
        "Generalisation regime"
    )

    metric_name = visible_metric_name(
        metric_label
    )

    axis.set_ylabel(
        f"{metric_name} change (T2 − T1)"
    )

    axis.set_ylim(
        *padded_limits(
            metric_seed[
                "value"
            ].to_numpy(dtype=float),
            include_zero=True,
            fraction=0.12,
        )
    )

    set_metric_title(
        axis,
        metric_label,
    )

    clean_axis(
        axis,
        grid_axis="y",
    )


def plot_panel_b(
    axis: plt.Axes,
    *,
    metric_label: str,
) -> None:
    metric_summary = panel_b_summary[
        panel_b_summary["metric_label"].eq(
            metric_label
        )
    ].copy()

    x = np.arange(
        len(REGIME_ORDER),
        dtype=float,
    )

    for task_label in TASK_ORDER:
        task_data = (
            metric_summary[
                metric_summary["task_label"].eq(
                    task_label
                )
            ]
            .set_index("regime_id")
            .loc[REGIME_ORDER]
            .reset_index()
        )

        median = task_data[
            "median"
        ].to_numpy(dtype=float)

        q1 = task_data[
            "q1"
        ].to_numpy(dtype=float)

        q3 = task_data[
            "q3"
        ].to_numpy(dtype=float)

        ci_low = task_data[
            "bootstrap_median_ci_low"
        ].to_numpy(dtype=float)

        ci_high = task_data[
            "bootstrap_median_ci_high"
        ].to_numpy(dtype=float)

        color = TASK_COLORS[
            task_label
        ]

        axis.fill_between(
            x,
            q1,
            q3,
            color=color,
            alpha=0.18,
            linewidth=0,
            zorder=1,
        )

        axis.plot(
            x,
            ci_low,
            color=color,
            linewidth=0.70,
            alpha=0.55,
            linestyle=(0, (2, 2)),
            zorder=2,
        )

        axis.plot(
            x,
            ci_high,
            color=color,
            linewidth=0.70,
            alpha=0.55,
            linestyle=(0, (2, 2)),
            zorder=2,
        )

        axis.plot(
            x,
            median,
            color=color,
            linewidth=2.4,
            marker="o",
            markersize=5.1,
            markeredgewidth=0.45,
            markeredgecolor=EDGE_COLOR,
            label=task_label,
            zorder=3,
        )

    axis.set_xticks(x)
    axis.set_xticklabels(
        [
            REGIME_LABELS[regime_id]
            for regime_id in REGIME_ORDER
        ]
    )

    axis.set_xlabel(
        "Generalisation regime"
    )

    metric_name = visible_metric_name(
        metric_label
    )

    axis.set_ylabel(metric_name)

    values = panel_b_seed.loc[
        panel_b_seed["metric_label"].eq(
            metric_label
        ),
        "value",
    ].to_numpy(dtype=float)

    if metric_label == "AP":
        axis.set_ylim(
            *padded_limits(
                values,
                lower_bound=0.0,
                upper_bound=1.0,
            )
        )
    else:
        axis.set_ylim(
            *padded_limits(
                values,
                lower_bound=-1.0,
                upper_bound=1.0,
            )
        )

    set_metric_title(
        axis,
        metric_label,
    )

    clean_axis(
        axis,
        grid_axis="y",
    )


def plot_panel_c(
    axis: plt.Axes,
    *,
    metric_label: str,
) -> None:
    metric_data = panel_c_summary[
        panel_c_summary["metric_label"].eq(
            metric_label
        )
    ].copy()

    y_positions = np.arange(
        len(EFFECT_ORDER)
    )[::-1]

    for y, effect_id in zip(
        y_positions,
        EFFECT_ORDER,
    ):
        row = metric_data[
            metric_data["effect_id"].eq(
                effect_id
            )
        ]

        if len(row) != 1:
            raise ValueError(
                f"Expected one Panel C row for "
                f"{metric_label}/{effect_id}."
            )

        row = row.iloc[0]
        color = EFFECT_COLORS[
            effect_id
        ]

        draw_interval(
            axis,
            y=float(y),
            median=float(row["median"]),
            q1=float(row["q1"]),
            q3=float(row["q3"]),
            ci_low=float(
                row[
                    "bootstrap_median_ci_low"
                ]
            ),
            ci_high=float(
                row[
                    "bootstrap_median_ci_high"
                ]
            ),
            color=color,
        )

        axis.text(
            float(row["median"]),
            float(y) + 0.20,
            format_effect_value(
                float(row["median"])
            ),
            fontsize=ANNOTATION_SIZE,
            fontweight="bold",
            color=color,
            ha="center",
            va="center",
        )

    axis.axvline(
        0.0,
        color=ZERO_COLOR,
        linewidth=0.85,
        linestyle=(0, (4, 3)),
        zorder=0,
    )

    values = np.concatenate(
        [
            metric_data[
                "bootstrap_median_ci_low"
            ].to_numpy(dtype=float),
            metric_data[
                "bootstrap_median_ci_high"
            ].to_numpy(dtype=float),
            np.array([0.0]),
        ]
    )

    axis.set_xlim(
        *padded_limits(
            values,
            include_zero=True,
            fraction=0.24,
        )
    )

    axis.set_ylim(
        -0.6,
        len(EFFECT_ORDER) - 0.4,
    )

    axis.set_yticks(y_positions)
    axis.set_yticklabels(
        [
            EFFECT_LABELS[effect_id]
            for effect_id in EFFECT_ORDER
        ]
    )

    metric_name = visible_metric_name(
        metric_label
    )

    axis.set_xlabel("")

    # Fewer labels and slightly smaller x tick text improve readability.
    axis.xaxis.set_major_locator(
        MaxNLocator(nbins=5)
    )
    axis.tick_params(
        axis="x",
        labelsize=7.8,
    )

    axis.text(
        0.02,
        0.04,
        "Lower",
        transform=axis.transAxes,
        fontsize=7.2,
        color=ZERO_COLOR,
        ha="left",
        va="bottom",
    )

    axis.text(
        0.98,
        0.04,
        "Higher",
        transform=axis.transAxes,
        fontsize=7.2,
        color=ZERO_COLOR,
        ha="right",
        va="bottom",
    )

    set_metric_title(
        axis,
        metric_label,
    )

    clean_axis(
        axis,
        grid_axis="x",
    )


def plot_panel_d(
    axis: plt.Axes,
    *,
    metric_label: str,
) -> None:
    metric_data = panel_d_summary[
        panel_d_summary["metric_label"].eq(
            metric_label
        )
    ].copy()

    y_positions = np.arange(
        len(MAGNITUDE_ORDER)
    )[::-1]

    for y, effect_id in zip(
        y_positions,
        MAGNITUDE_ORDER,
    ):
        row = metric_data[
            metric_data["effect_id"].eq(
                effect_id
            )
        ]

        if len(row) != 1:
            raise ValueError(
                f"Expected one Panel D row for "
                f"{metric_label}/{effect_id}."
            )

        row = row.iloc[0]
        color = EFFECT_COLORS[
            effect_id
        ]

        draw_interval(
            axis,
            y=float(y),
            median=float(row["median"]),
            q1=float(row["q1"]),
            q3=float(row["q3"]),
            ci_low=float(
                row[
                    "bootstrap_median_ci_low"
                ]
            ),
            ci_high=float(
                row[
                    "bootstrap_median_ci_high"
                ]
            ),
            color=color,
        )

        axis.text(
            float(row["median"]),
            float(y) + 0.20,
            format_effect_value(
                float(row["median"])
            ),
            fontsize=ANNOTATION_SIZE,
            fontweight="bold",
            color=color,
            ha="center",
            va="center",
        )

    axis.set_ylim(
        -0.6,
        len(MAGNITUDE_ORDER) - 0.4,
    )

    axis.set_yticks(y_positions)
    axis.set_yticklabels(
        [
            EFFECT_LABELS[effect_id]
            for effect_id in MAGNITUDE_ORDER
        ]
    )

    metric_name = visible_metric_name(
        metric_label
    )

    axis.set_xlabel(
        f"Absolute change in {metric_name}"
    )

    axis.set_xlim(
        0.0,
        padded_limits(
            metric_data[
                "bootstrap_median_ci_high"
            ].to_numpy(dtype=float),
            lower_bound=0.0,
            fraction=0.18,
        )[1],
    )

    set_metric_title(
        axis,
        metric_label,
    )

    clean_axis(
        axis,
        grid_axis="x",
    )


## 6. Assemble Figure 2

In [ ]:

figure = plt.figure(
    figsize=(
        FIGURE_WIDTH_INCHES,
        FIGURE_HEIGHT_INCHES,
    ),
    constrained_layout=False,
)

subfigures = figure.subfigures(
    nrows=2,
    ncols=2,
    width_ratios=[1.0, 1.0],
    height_ratios=[1.0, 1.0],
    wspace=0.08,
    hspace=0.10,
)

subfigure_a = subfigures[0, 0]
subfigure_c = subfigures[0, 1]
subfigure_b = subfigures[1, 0]
subfigure_d = subfigures[1, 1]


def add_subfigure_header(
    subfigure,
    *,
    letter: str,
    title: str,
) -> None:
    subfigure.text(
        0.00,
        0.985,
        letter,
        fontsize=PANEL_LETTER_SIZE,
        fontweight="bold",
        ha="left",
        va="top",
    )

    subfigure.text(
        0.065,
        0.985,
        title,
        fontsize=PANEL_TITLE_SIZE,
        fontweight="bold",
        ha="left",
        va="top",
    )


# ------------------------------------------------------------------
# Panel A
# Same internal margins as Panel C.
# ------------------------------------------------------------------
axis_a_ap, axis_a_mcc = subfigure_a.subplots(
    nrows=1,
    ncols=2,
    gridspec_kw={
        "wspace": 0.30,
    },
)

subfigure_a.subplots_adjust(
    left=0.12,
    right=0.98,
    top=0.84,
    bottom=0.21,
)

plot_panel_a(
    axis_a_ap,
    metric_label="AP",
)

plot_panel_a(
    axis_a_mcc,
    metric_label="MCC",
)

add_subfigure_header(
    subfigure_a,
    letter="A",
    title="Effect of negative-class construction",
)


# ------------------------------------------------------------------
# Panel C
# Shared xlabel and legend live at SubFigure level.
# ------------------------------------------------------------------
axis_c_ap, axis_c_mcc = subfigure_c.subplots(
    nrows=1,
    ncols=2,
    gridspec_kw={
        "wspace": 0.26,
    },
)

subfigure_c.subplots_adjust(
    left=0.17,
    right=0.98,
    top=0.84,
    bottom=0.27,
)

plot_panel_c(
    axis_c_ap,
    metric_label="AP",
)

plot_panel_c(
    axis_c_mcc,
    metric_label="MCC",
)

axis_c_mcc.set_yticklabels([])

add_subfigure_header(
    subfigure_c,
    letter="C",
    title="Effect size of methodological decisions",
)

subfigure_c.supxlabel(
    "Change in performance",
    x=0.58,
    y=0.145,
    fontsize=AXIS_LABEL_SIZE,
)

interval_legend = [
    Line2D(
        [0],
        [0],
        color=EDGE_COLOR,
        linewidth=1.2,
        label="Bootstrap 95% CI",
    ),
    Line2D(
        [0],
        [0],
        color=EDGE_COLOR,
        linewidth=4.7,
        label="IQR",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="none",
        markerfacecolor="#B7BDC5",
        markeredgecolor=EDGE_COLOR,
        markersize=6.5,
        label="Median",
    ),
]

subfigure_c.legend(
    handles=interval_legend,
    loc="lower center",
    bbox_to_anchor=(0.58, 0.025),
    bbox_transform=subfigure_c.transSubfigure,
    ncol=3,
    columnspacing=1.8,
    handlelength=2.1,
    borderaxespad=0.0,
    frameon=False,
)


# ------------------------------------------------------------------
# Panel B
# Same internal plot dimensions as Panel D.
# Shared xlabel and legend live at SubFigure level.
# ------------------------------------------------------------------
axis_b_ap, axis_b_mcc = subfigure_b.subplots(
    nrows=2,
    ncols=1,
    gridspec_kw={
        "hspace": 0.16,
    },
)

subfigure_b.subplots_adjust(
    left=0.11,
    right=0.98,
    top=0.84,
    bottom=0.23,
)

plot_panel_b(
    axis_b_ap,
    metric_label="AP",
)

plot_panel_b(
    axis_b_mcc,
    metric_label="MCC",
)

axis_b_ap.set_xlabel("")
axis_b_ap.set_xticklabels([])
axis_b_ap.tick_params(
    axis="x",
    length=0,
)

axis_b_mcc.set_xlabel("")
axis_b_mcc.tick_params(
    axis="x",
    pad=5,
)

add_subfigure_header(
    subfigure_b,
    letter="B",
    title="Effect of partition strategy and threshold",
)

subfigure_b.supxlabel(
    "Generalisation regime",
    x=0.55,
    y=0.125,
    fontsize=AXIS_LABEL_SIZE,
)

task_legend = [
    Line2D(
        [0],
        [0],
        color=TASK_COLORS["T1"],
        linewidth=2.4,
        marker="o",
        markersize=5.2,
        markeredgecolor=EDGE_COLOR,
        markeredgewidth=0.45,
        label="T1: toxic negatives",
    ),
    Line2D(
        [0],
        [0],
        color=TASK_COLORS["T2"],
        linewidth=2.4,
        marker="o",
        markersize=5.2,
        markeredgecolor=EDGE_COLOR,
        markeredgewidth=0.45,
        label="T2: background negatives",
    ),
]

subfigure_b.legend(
    handles=task_legend,
    loc="lower center",
    bbox_to_anchor=(0.55, 0.015),
    bbox_transform=subfigure_b.transSubfigure,
    ncol=2,
    columnspacing=2.2,
    handlelength=2.1,
    borderaxespad=0.0,
    frameon=False,
)


# ------------------------------------------------------------------
# Panel D
# The same top and bottom margins as Panel B preserve equal plot size.
# ------------------------------------------------------------------
axis_d_ap, axis_d_mcc = subfigure_d.subplots(
    nrows=2,
    ncols=1,
    gridspec_kw={
        "hspace": 0.16,
    },
)

subfigure_d.subplots_adjust(
    left=0.17,
    right=0.98,
    top=0.84,
    bottom=0.23,
)

plot_panel_d(
    axis_d_ap,
    metric_label="AP",
)

plot_panel_d(
    axis_d_mcc,
    metric_label="MCC",
)

axis_d_ap.set_xlabel("")
axis_d_ap.set_xticklabels([])
axis_d_ap.tick_params(
    axis="x",
    length=0,
)

add_subfigure_header(
    subfigure_d,
    letter="D",
    title="Methodological effects versus model variation",
)

figure.savefig(
    FIGURE_PNG_PATH,
    dpi=RASTER_DPI,
    bbox_inches="tight",
    facecolor=BACKGROUND_COLOR,
)

figure.savefig(
    FIGURE_PDF_PATH,
    bbox_inches="tight",
    facecolor=BACKGROUND_COLOR,
)

figure.savefig(
    FIGURE_SVG_PATH,
    bbox_inches="tight",
    facecolor=BACKGROUND_COLOR,
)

plt.show()


## 7. Confirm outputs

In [ ]:

for path in [
    FIGURE_PNG_PATH,
    FIGURE_PDF_PATH,
    FIGURE_SVG_PATH,
]:
    if not path.is_file():
        raise RuntimeError(
            f"Expected output was not created: {path}"
        )

    print(
        f"{path.relative_to(REPO_ROOT)} "
        f"({path.stat().st_size / 1024:.1f} KiB)"
    )


## 8. Figure manifest

In [ ]:

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while chunk := handle.read(
            chunk_size
        ):
            digest.update(chunk)

    return digest.hexdigest()


def package_version(
    package_name: str,
) -> str:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not-installed"


def git_value(
    arguments: list[str],
) -> str | None:
    try:
        completed = subprocess.run(
            ["git", *arguments],
            cwd=REPO_ROOT,
            check=True,
            capture_output=True,
            text=True,
        )

        return completed.stdout.strip()

    except (
        FileNotFoundError,
        subprocess.CalledProcessError,
    ):
        return None


manifest = {
    "schema_version": "1.0",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "figure": "Figure 2 v8",
    "layout": {
        "outer_layout": "2x2 SubFigure grid",
        "top_row": ["A", "C"],
        "bottom_row": ["B", "D"],
        "equal_outer_panel_dimensions": True,
        "panel_b_shared_xlabel": (
            "SubFigure-level Generalisation regime"
        ),
        "panel_b_legend": (
            "SubFigure-level legend below shared xlabel"
        ),
        "panel_c_shared_xlabel": (
            "SubFigure-level Change in performance"
        ),
        "panel_c_legend": (
            "SubFigure-level legend below shared xlabel"
        ),
    },
    "visual_systems": {
        "tasks": TASK_COLORS,
        "regimes": REGIME_COLORS,
        "methodological_decisions": (
            EFFECT_COLORS
        ),
    },
    "outputs": {
        "png": {
            "path": str(
                FIGURE_PNG_PATH.relative_to(
                    REPO_ROOT
                )
            ),
            "sha256": sha256_file(
                FIGURE_PNG_PATH
            ),
        },
        "pdf": {
            "path": str(
                FIGURE_PDF_PATH.relative_to(
                    REPO_ROOT
                )
            ),
            "sha256": sha256_file(
                FIGURE_PDF_PATH
            ),
        },
        "svg": {
            "path": str(
                FIGURE_SVG_PATH.relative_to(
                    REPO_ROOT
                )
            ),
            "sha256": sha256_file(
                FIGURE_SVG_PATH
            ),
        },
    },
    "inputs": {
        name: {
            "path": str(
                path.relative_to(
                    REPO_ROOT
                )
            ),
            "sha256": sha256_file(
                path
            ),
        }
        for name, path in INPUT_PATHS.items()
    },
    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": package_version(
            "numpy"
        ),
        "pandas": package_version(
            "pandas"
        ),
        "matplotlib": package_version(
            "matplotlib"
        ),
    },
    "git": {
        "commit": git_value(
            ["rev-parse", "HEAD"]
        ),
        "status_porcelain": git_value(
            ["status", "--porcelain"]
        ),
    },
}

with FIGURE_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    handle.write("\n")

print(
    "Figure manifest:",
    FIGURE_MANIFEST_PATH.relative_to(
        REPO_ROOT
    ),
)
